# Base pricing env

> Base environment with some basic funcitons

In [ ]:
#| default_exp envs.pricing.base

In [ ]:
#| hide
from nbdev.showdoc import *

In [2]:
#| export
from abc import ABC, abstractmethod
from typing import Union, Tuple, List

from ddopai.envs.base import BaseEnvironment
from ddopai.utils import Parameter, MDPInfo
from ddopai.dataloaders.base import BaseDataLoader
from ddopai.loss_functions import pinball_loss

import gymnasium as gym

import numpy as np
import time

In [ ]:
#| export
class BasePricingEnv(BaseEnvironment):
    """
    Base class for inventory management environments. This class inherits from BaseEnvironment.
    
    """

    def __init__(self, 

        ## Parameters for Base env:
        mdp_info: MDPInfo, #
        postprocessors: list[object] | None = None,  # default is empty list
        mode: str = "online", # additional mode for the pricing environment TODO: add online mode to training loop
        return_truncation: str = True, # whether to return a truncated condition in step function
        dataloader: BaseDataLoader = None, # dataloader for the environment
        
        alpha: Union[float, np.ndarray] = 1, # market size parameter
        beta: Union[float, np.ndarray] = 1, # price sensitivity parameter
        horizon_train: int = 100 # horizon for the online learning TODO: check if it can be renamed to horizon
        ) -> None:

        self.dataloader = dataloader
        
        self.set_param("alpha", alpha, shape=(self.nun_SKUs[0],), new=True)
        self.set_param("beta", beta, shape=(self.nun_SKUs[0],), new=True)
        
        # TODO: check in the base env if train_horizon is needed 
        super().__init__(mdp_info=mdp_info, postprocessors = postprocessors,  mode = mode, return_truncation=return_truncation, horizon_train=horizon_train)
    
    def set_observation_space(self,
                            shape: tuple, # shape of the dataloader features
                            low: Union[np.ndarray, float] = -np.inf, # lower bound of the observation space
                            high: Union[np.ndarray, float] = np.inf, # upper bound of the observation space
                            samples_dim_included = True # whether the first dimension of the shape input is the number of samples
                            ) -> None:
        
        '''
        Set the observation space of the environment.
        This is a standard function for simple observation spaces. For more complex observation spaces,
        this function should be overwritten. Note that it is assumped that the first dimension
        is n_samples that is not relevant for the observation space.

        '''

        # To handle cases when no external information is available (e.g., parametric NV)
        
        if shape is None:
            self.observation_space = None

        else:
            if not isinstance(shape, tuple):
                raise ValueError("Shape must be a tuple.")
            
            if samples_dim_included:
                shape = shape[1:] # assumed that the first dimension is the number of samples

            self.observation_space = gym.spaces.Box(low=low, high=high, shape=shape, dtype=np.float32)

    def set_action_space(self,
                            shape: tuple, # shape of the dataloader target
                            low: Union[np.ndarray, float] = -np.inf, # lower bound of the observation space
                            high: Union[np.ndarray, float] = np.inf, # upper bound of the observation space
                            samples_dim_included = True # whether the first dimension of the shape input is the number of samples
                            ) -> None:
        
        '''
        Set the action space of the environment.
        This is a standard function for simple action spaces. For more complex action spaces,
        this function should be overwritten. Note that it is assumped that the first dimension
        is n_samples that is not relevant for the action space.
        '''

        if not isinstance(shape, tuple):
            raise ValueError("Shape must be a tuple.")
        
        if samples_dim_included:
            shape = shape[1:] # assumed that the first dimension is the number of samples

        self.action_space = gym.spaces.Box(low=low, high=high, shape=shape, dtype=np.float32)
    
    def get_observation(self):
        
        """
        Return the current observation. This function is for the online learning case it will return only the state,
        this function should be overwritten.

        """

        X_item,  = self.dataloader[self.index]

        return X_item
    
    def get_demand_response(self, action):
            
            """
            Return the demand and the reward for the current action. This function should be overwritten.
            TODO: add the tuple call to the pricing dataloader
            """
            Y_item, epsilon = self.dataloader[self.index, action]
            return Y_item, epsilon
    def reset(self,
        start_index: int | str = None, # index to start from
        state: np.ndarray = None # initial state
        ) -> Tuple[np.ndarray, bool]:

        """
        Reset function for the Newsvendor problem. It will return the first observation and demand.
        For val and test modes, it will by default reset to 0, while for the train mode it depends
        on the paramter "horizon_train" whether a random point in the training data is selected or 0
        """

        truncated = self.reset_index(start_index)



        observation, self.demand = self.get_observation()
        
        return observation
